In [0]:
%sql

-- 1. Recebe os enderecos ficticios de um ERP
-- 2. Converte o JSON e transforma o array em linhas
-- 3. Deixa os dados disponiveis para as proximas celulas

CREATE OR REPLACE TEMP VIEW vw_enderecos_entrada AS

WITH origem AS (
    SELECT
        1001 AS id_cliente,
        'ERP_LEGADO' AS sistema_origem,
        '{
          "enderecos": [
            {
              "tipo": "RESIDENCIAL",
              "original": " ",
              "cep": "08717-260",
              "numero": "123"
            },
            {
              "tipo": "ENTREGA",
              "original": "nao informado",
              "cep": "08717260",
              "numero": "250"
            }
          ]
        }' AS json_origem
),

dados AS (
    SELECT
        id_cliente,
        sistema_origem,
        FROM_JSON(
            json_origem,
            'STRUCT<enderecos:ARRAY<STRUCT<
                tipo:STRING,
                original:STRING,
                cep:STRING,
                numero:STRING
            >>>'
        ) AS payload
    FROM origem
)

SELECT
    d.id_cliente,
    d.sistema_origem,
    e.tipo AS tipo_endereco,
    e.original AS endereco_original,
    REGEXP_REPLACE(e.cep, '[^0-9]', '') AS cep,
    e.numero
FROM dados d
LATERAL VIEW EXPLODE(d.payload.enderecos) x AS e;


In [0]:
# Busca os CEPs distintos recebidos do sistema de origem
ceps = spark.sql("""
    SELECT DISTINCT cep
    FROM vw_enderecos_entrada
    WHERE cep RLIKE '^[0-9]{8}$'
""").collect()

# Atualiza o cache consultando as APIs quando necessário
for linha in ceps:
    cep = linha["cep"]

    dbutils.notebook.run(
        "../src/01_api_viaCep", 120, {"cep": cep}
    )

    dbutils.notebook.run(
        "../src/02_api_brasilapi", 120, {"cep": cep}
    )

# Chama a função SQL e prepara os endereços padronizados
spark.sql("""
    CREATE OR REPLACE TEMP VIEW vw_enderecos_padronizados AS
    SELECT
        e.id_cliente,
        e.sistema_origem,
        e.tipo_endereco,
        e.endereco_original,
        f.cep,
        f.numero,
        f.logradouro,
        f.bairro,
        f.cidade,
        f.uf,
        f.endereco_completo,
        f.fonte,
        f.status_dq,
        CAST(f.percentual_completude AS DOUBLE)
            AS percentual_completude,
        CURRENT_TIMESTAMP() AS data_processamento

    FROM vw_enderecos_entrada e,
    LATERAL geocep.default.Fcn_Logradouro(
        e.cep, e.numero
    ) AS f
""")

display(spark.table("vw_enderecos_padronizados"))

In [0]:
%sql
INSERT INTO geocep.default.dim_endereco
SELECT *
FROM vw_enderecos_padronizados;

In [0]:
%sql
select * from geocep.default.dim_endereco